# Results Analysis — LiteCARE-RAGent
Final evaluation figures and tables for the thesis.

**Requires**: Completed training runs from scripts 04–06.  
Reads result CSV files from `results/` directory.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.visualization import (
    plot_confusion_matrix, plot_roc_curves, plot_calibration_diagram,
    plot_uncertainty_distribution, plot_ablation_bars,
    plot_size_vs_accuracy, plot_loso_f1_per_subject
)

RESULTS_DIR = Path('../results')
FIGS_DIR    = Path('../results/figures')
FIGS_DIR.mkdir(parents=True, exist_ok=True)

## 1. WESAD LOSO — Full Model

In [ ]:
# Load per-fold results
loso_csv = RESULTS_DIR / 'wesad_loso' / 'full' / 'loso_results.csv'
if loso_csv.exists():
    df_loso = pd.read_csv(loso_csv)
    print(df_loso.to_string(index=False))

    print('\n--- Summary ---')
    for col in ['accuracy', 'f1', 'auroc', 'ece', 'abstention_rate']:
        if col in df_loso.columns:
            print(f'  {col:20s}: {df_loso[col].mean():.4f} ± {df_loso[col].std():.4f}')
else:
    print('Run script 04_train.py first.')

In [ ]:
# LOSO F1 per subject bar chart
if loso_csv.exists():
    fold_results = df_loso[['test_subject', 'f1']].rename(
        columns={'test_subject': 'test_subject', 'f1': 'f1'}
    ).to_dict('records')

    plot_loso_f1_per_subject(
        fold_results,
        title='WESAD LOSO — F1 per Subject (Full Model)',
        out_path=FIGS_DIR / 'loso_f1_per_subject.png'
    )

## 2. Ablation Study

In [ ]:
ablation_csv = RESULTS_DIR / 'ablation' / 'ablation_summary.csv'
if ablation_csv.exists():
    df_abl = pd.read_csv(ablation_csv, index_col=0)
    print(df_abl.to_string())

    abl_dict = {}
    for variant, row in df_abl.iterrows():
        abl_dict[variant] = row.to_dict()

    plot_ablation_bars(
        abl_dict,
        metric='f1',
        title='Ablation Study — WESAD LOSO F1',
        out_path=FIGS_DIR / 'ablation_f1.png'
    )
else:
    print('Run script 06_ablation_study.py first.')

## 3. Model Size vs. F1 (All Models)

In [ ]:
# Manually populate with results from training
# Replace values with actual numbers after training
model_info = {
    'LiteTCN-SE (full)':   {'size_mb': 2.1,  'f1': 0.0, 'params': 450000},
    'LiteTCN-SE (no_se)':  {'size_mb': 1.9,  'f1': 0.0, 'params': 420000},
    'BiLSTM':              {'size_mb': 3.5,  'f1': 0.0, 'params': 800000},
    'DilatedCNN':          {'size_mb': 1.8,  'f1': 0.0, 'params': 380000},
    'Random Forest':       {'size_mb': 0.5,  'f1': 0.0, 'params': 0},
}
print('Update model_info with actual training results before plotting.')
# plot_size_vs_accuracy(model_info, out_path=FIGS_DIR / 'size_vs_f1.png')

## 4. Cross-Dataset Results

In [ ]:
cross_csv = RESULTS_DIR / 'cross_dataset' / 'cross_eval_results.csv'
if cross_csv.exists():
    df_cross = pd.read_csv(cross_csv, index_col=0)
    print(df_cross.to_string())
else:
    print('Run script 05_cross_dataset_eval.py first.')

## 5. Uncertainty Analysis

In [ ]:
# Load uncertainty data if saved during evaluation
unc_path = RESULTS_DIR / 'wesad_loso' / 'full' / 'uncertainties.npz'
if unc_path.exists():
    arr = np.load(unc_path)
    y_true = arr['y_true']
    y_pred = arr['y_pred']
    uncerts = arr['uncertainties']
    tau = float(arr.get('tau', 0.5))

    plot_uncertainty_distribution(
        uncertainties=uncerts,
        y_true=y_true,
        y_pred=y_pred,
        tau=tau,
        title='Uncertainty Distribution — WESAD LOSO (Full Model)',
        out_path=FIGS_DIR / 'uncertainty_distribution.png'
    )

    # Calibration diagram
    probs_path = RESULTS_DIR / 'wesad_loso' / 'full' / 'probabilities.npz'
    if probs_path.exists():
        parr = np.load(probs_path)
        plot_calibration_diagram(
            parr['y_true'], parr['y_prob'],
            title='Calibration Diagram — WESAD LOSO',
            out_path=FIGS_DIR / 'calibration.png'
        )
else:
    print('Uncertainty data not yet available. Run training first.')

## 6. RAG-Mini Demo

In [ ]:
from src.rag_mini.retrieval import load_knowledge_base, retrieve, format_intervention

kb = load_knowledge_base()

contexts = ['workplace', 'driving', 'wearable', 'general']
for ctx in contexts:
    print(f'\n=== Context: {ctx} ===')
    results = retrieve(stress_state='stress', context=ctx, activity_level='sedentary', top_k=3, kb=kb)
    for iv in results:
        print(f'  [{iv["type"]}/{iv["intensity"]}] {format_intervention(iv)}')